In [ ]:
import ROOT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tools import scale_out as so
import emm
import emm.bias as bias

In [ ]:
# # Load data
x = ROOT.RooRealVar("x", "Diphoton Mass [GeV]", 500, 10_000)
data_tree = emm.data.get_diphoton_data(sort_and_index=True, tree=True)
data = ROOT.RooDataSet("mgg", "mgg", ROOT.RooArgSet(x), ROOT.RooFit.Import(data_tree))
n = data.numEntries()
data_mean = data.mean(x)
print(f"Data mean: {data_mean}")

In [ ]:
# Configuration
x_min, x_max = 500, 3500
x_grid = np.linspace(x_min, x_max, 1000)

# Default
n_seeds = 50
n_toys_per_seed = 100

# Generate seeds
default_seed = 42
np.random.seed(default_seed)
seeds = np.random.randint(0, 10000, size=n_seeds)

In [ ]:
# Set up models
toy_models = [
    emm.f1(x),
    emm.f2(x),
    emm.f3(x),
    emm.f4(x),
]

model_primitives = [
    emm.f1,
    emm.f2,
    emm.f3,
    emm.f4,
]
for k in [2, 3, 4]:
    model_primitives.append(
        emm.models.ModelPrimitive(
            emm.ExponentialMixtureModel,
            k,
            data_mean=700,
            name=f"ExponentialMixture-{k}",
        )
    )

print(f"Will be submitting {n_seeds * len(toy_models)} tasks, with {n_toys_per_seed} toys fitting {len(model_primitives)} models.")

# Fit toy models to data
for toy_model in toy_models:
    print(f"Fitting toy model: {toy_model.name}")
    toy_model.pdf.fitTo(data)

In [ ]:
# Run jobs
tasks = []
for seed in seeds:
    for toy_model in toy_models:
        tasks.append(so.Task(bias.run_bias_fits, x, toy_model, model_primitives, seed, n_toys_per_seed, n, x_grid))

# _ = so.run_tasks(
#     tasks,
#     use_condor=True,
#     condor_job_name="bias",
#     env_wrapper=so.run_in_mamba,
#     clear_logs=True
# )

In [ ]:
# Load and combine results
results = bias.get_bias_results(
    toy_models,
    seeds,
    n_toys_per_seed,
    n,
 )

In [ ]:
_ = bias.plot_bias(
    x,
    toy_models,
    results,
    x_grid,
    range=(500, 3500),
)